In [11]:
from datasets import load_dataset

In [12]:
# System message for the assistant
system_message = """You are a text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA."""

# User prompt that combines the user query and the schema
user_prompt = """Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.

<SCHEMA>
{context}
</SCHEMA>

<USER_QUERY>
{question}
</USER_QUERY>
"""
def create_conversation(sample):
  return {
    "messages": [
      # {"role": "system", "content": system_message},
      {"role": "user", "content": user_prompt.format(question=sample["sql_prompt"], context=sample["sql_context"])},
      {"role": "assistant", "content": sample["sql"]}
    ]
  }

# Load dataset from the hub
dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql", split="train")
dataset = dataset.shuffle().select(range(12500))

In [3]:
dataset

Dataset({
    features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
    num_rows: 12500
})

In [ ]:
dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=True)

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

ArrowInvalid: cannot mix list and non-list, non-null values

In [ ]:
# Convert dataset to OAI messages
dataset = dataset.map(create_conversation, remove_columns=dataset.features,batched=False)
# split dataset into 10,000 training samples and 2,500 test samples
dataset = dataset.train_test_split(test_size=2500/12500)

# Print formatted user prompt
print(dataset["train"][345]["messages"][1]["content"])

In [1]:
batch = {
    "sql_prompt":  ["SELECT … FROM …",  "SELECT …",  "SELECT …"],
    "sql_context": ["CREATE TABLE …",    "CREATE TABLE …", "CREATE TABLE …"],
    "sql":         ["SELECT …",          "SELECT …",       "SELECT …"],
    # …(다른 컬럼도 같은 길이의 리스트)
}

In [ ]:
# 3개의 list를 zip하여 각 요소를 출력
for q, ctx, sql in zip(batch["sql_prompt"],
                           batch["sql_context"],
                           batch["sql"]):
    print(f"Question: {q}")
    print(f"Context: {ctx}")
    print(f"SQL: {sql}")

Question: SELECT … FROM …
Context: CREATE TABLE …
SQL: SELECT …
Question: SELECT …
Context: CREATE TABLE …
SQL: SELECT …
Question: SELECT …
Context: CREATE TABLE …
SQL: SELECT …


In [3]:
from datasets import load_dataset

dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql", split="train")
dataset = dataset.shuffle(seed=42).select(range(10))   # 샘플 10개만

# iter(batch_size=3) → 한 번에 3행씩 딕셔너리 형태로 리턴
first_batch = next(dataset.iter(batch_size=3))
print("type:", type(first_batch))

type: <class 'dict'>


In [4]:
first_batch

{'id': [66411, 35885, 74018],
 'domain': ['fitness industry', 'blockchain', 'media'],
 'domain_description': ['Workout data, membership demographics, wearable technology metrics, and wellness trends.',
  'Comprehensive data on smart contracts, decentralized applications, digital assets, and regulatory frameworks in blockchain.',
  'Media data on content diversity, media literacy, disinformation detection, and media representation.'],
 'sql_complexity': ['single join', 'aggregation', 'single join'],
 'sql_complexity_description': ['only one join (specify inner, outer, cross)',
  'aggregation functions (COUNT, SUM, AVG, MIN, MAX, etc.), and HAVING clause',
  'only one join (specify inner, outer, cross)'],
 'sql_task_type': ['analytics and reporting',
  'analytics and reporting',
  'analytics and reporting'],
 'sql_task_type_description': ['generating reports, dashboards, and analytical insights',
  'generating reports, dashboards, and analytical insights',
  'generating reports, dashboar

In [5]:
for k, v in first_batch.items():
    print(f"{k:12} | len={len(v)} | type={type(v)}")
    print("  샘플:", v[0], "\n")

id           | len=3 | type=<class 'list'>
  샘플: 66411 

domain       | len=3 | type=<class 'list'>
  샘플: fitness industry 

domain_description | len=3 | type=<class 'list'>
  샘플: Workout data, membership demographics, wearable technology metrics, and wellness trends. 

sql_complexity | len=3 | type=<class 'list'>
  샘플: single join 

sql_complexity_description | len=3 | type=<class 'list'>
  샘플: only one join (specify inner, outer, cross) 

sql_task_type | len=3 | type=<class 'list'>
  샘플: analytics and reporting 

sql_task_type_description | len=3 | type=<class 'list'>
  샘플: generating reports, dashboards, and analytical insights 

sql_prompt   | len=3 | type=<class 'list'>
  샘플: How many members attended each type of class in April 2022? 

sql_context  | len=3 | type=<class 'list'>
  샘플: CREATE TABLE Members (MemberID INT, Age INT, Gender VARCHAR(10), MembershipType VARCHAR(20)); INSERT INTO Members (MemberID, Age, Gender, MembershipType) VALUES (1, 35, 'Female', 'Premium'), (2, 45, 

In [ ]:
from itertools import count
batch_no = count(0)


def debug_batch(batch):
    idx = next(batch_no)
    print(f"\n🟦 Batch {idx} (size={len(batch['sql_prompt'])})")
    print("   sql_prompt :", batch['sql_prompt'][0])
    print("   sql_context:", batch['sql_context'][0][:80], "…")
    print("   sql        :", batch['sql'][0])
    
    return batch   # 그대로 돌려줘야 map()이 지속

# 캐시를 쓰면 함수가 안 돌 수도 있으니 load_from_cache_file=False
_ = (
    dataset
    .shuffle(seed=0).select(range(6))         # 작은 샘플
    .map(
        debug_batch,
        batched=True,
        batch_size=2,                         # 2행씩, 디폴트는 1000
        load_from_cache_file=False
    )
)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]


🟦 Batch 0 (size=2)
   sql_prompt : Add a new restorative justice program to the "programs" table
   sql_context: CREATE TABLE programs (id INT, name VARCHAR(50), type VARCHAR(20), location VARC …
   sql        : INSERT INTO programs (id, name, type, location) VALUES (3002, 'Neighborhood Healing', 'Restorative Justice', 'San Francisco');

🟦 Batch 1 (size=2)
   sql_prompt : How many female and male authors have contributed to the articles?
   sql_context: CREATE TABLE Authors (id INT PRIMARY KEY, name VARCHAR(100), gender VARCHAR(10)) …
   sql        : SELECT gender, COUNT(author_id) as num_authors FROM ArticleAuthors aa JOIN Authors a ON aa.author_id = a.id GROUP BY gender;

🟦 Batch 2 (size=2)
   sql_prompt : Create a table named 'player_achievements'
   sql_context: CREATE TABLE player_achievements (player_id INT, achievement_name VARCHAR(255),  …
   sql        : CREATE TABLE player_achievements (player_id INT, achievement_name VARCHAR(255), achievement_date DATE);


In [8]:
dataset.features

{'id': Value(dtype='int32', id=None),
 'domain': Value(dtype='string', id=None),
 'domain_description': Value(dtype='string', id=None),
 'sql_complexity': Value(dtype='string', id=None),
 'sql_complexity_description': Value(dtype='string', id=None),
 'sql_task_type': Value(dtype='string', id=None),
 'sql_task_type_description': Value(dtype='string', id=None),
 'sql_prompt': Value(dtype='string', id=None),
 'sql_context': Value(dtype='string', id=None),
 'sql': Value(dtype='string', id=None),
 'sql_explanation': Value(dtype='string', id=None)}

In [9]:
dataset.column_names

['id',
 'domain',
 'domain_description',
 'sql_complexity',
 'sql_complexity_description',
 'sql_task_type',
 'sql_task_type_description',
 'sql_prompt',
 'sql_context',
 'sql',
 'sql_explanation']

In [ ]:
def create_conversation(batch):
    """
    chat혹은 conversation이라 불리는 형식으로 변환하는 함수

    role : 대화 참여자의 형식을 명확히 구분. 
    message는 리스트 형태 -> 멀티턴 대화까지 지원

    
    batch 처리의 경우 다음처럼 들어옴.
    각 키는 모두 N개의 길이를 가지는 리스트
        batch = {
        "sql_prompt":  ["SELECT … FROM …",  "SELECT …",  "SELECT …"],
        "sql_context": ["CREATE TABLE …",    "CREATE TABLE …", "CREATE TABLE …"],
        "sql":         ["SELECT …",          "SELECT …",       "SELECT …"],
        # …(다른 컬럼도 같은 길이의 리스트)
    }

    """
    conversations = []
    for q, ctx, sql in zip(batch["sql_prompt"], batch["sql_context"], batch["sql"]):
        conversations.append([
            {"role": "user",
             "content": user_prompt.format(question=q, context=ctx)},
            {"role": "assistant", "content": sql}
        ])
    return {"messages": conversations}  # key마다 리스트 길이가 동일해야 함!

# batched=True일 때는 지정한 함수안에 batch 단위로 들어감
# 이 batch는 dict-of-list 형태로 원래 데이터셋의 각 컬럼이 리스트로 묶인 형태
new_dataset = dataset.map(
    create_conversation,
    batched=True,
    remove_columns=dataset.features,
    # batch_size=1000  # 필요하면 직접 지정
)

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

In [14]:
new_dataset

Dataset({
    features: ['messages'],
    num_rows: 12500
})

In [16]:
# 분할
from datasets import DatasetDict

# ❶ train : val : test = 80 : 10 : 10  예시
split = new_dataset.train_test_split(test_size=2000/12500, seed=42)          # 먼저 train vs temp(=val+test)

temp = split.pop("test")                               # temp = 20 %
val_test = temp.train_test_split(test_size=0.5, seed=42)


dataset_dict = DatasetDict({
    "train": split["train"],
    "validation": val_test["train"],                   # 10 %
    "test": val_test["test"],                          # 10 %
})

In [17]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 10500
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 1000
    })
})

In [ ]:
from huggingface_hub import login, create_repo


hf_token = ""
login(token=hf_token)

create_repo("ty-kim/sql_translator", repo_type="dataset", exist_ok=True)

# push to hub
dataset.push_to_hub("ty-kim/sql_translator") # max_shard_size="500MB"

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/ty-kim/sql_translator/commit/178cb9ec9c6510e534cb355990e4db5987bc9df0', commit_message='Upload dataset', commit_description='', oid='178cb9ec9c6510e534cb355990e4db5987bc9df0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ty-kim/sql_translator', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ty-kim/sql_translator'), pr_revision=None, pr_num=None)

In [ ]:
dataset_dict.push_to_hub("ty-kim/sql_translator",
                    private=True,            # 공개로 올리려면 False
                    max_shard_size="1GB",    # → 파일 하나가 1 GB 넘으면 자동으로 shard
                    token=True,               # login() 대신 토큰 문자열 직접 넣어도 됨
                    commit_message="Reup",
                    ) # max_shard_size="500MB"

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/713 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/ty-kim/sql_translator/commit/ce3d9fdeacffc3ed7b15786e378c9bffdd55ecfb', commit_message='Upload dataset', commit_description='', oid='ce3d9fdeacffc3ed7b15786e378c9bffdd55ecfb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ty-kim/sql_translator', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ty-kim/sql_translator'), pr_revision=None, pr_num=None)

In [28]:
from huggingface_hub import DatasetCard, DatasetCardData

card_data = DatasetCardData(
    language=["en"],                # ISO 639-1 코드
    license="mit",                  # ex) mit / apache-2.0 …
    pretty_name="Text-to-SQL Conversations",
    size_categories=["10K<n<100K"], # 선택
    tags=["text-to-sql", "synthetic"]
)

# ▶ 카드 템플릿 생성
card = DatasetCard.from_template(card_data=card_data)   # 템플릿 채워진 객체
card.save("README.md")                                  # 로컬 파일 생성

# 필요하다면 push
card.push_to_hub("ty-kim/sql_translator")

CommitInfo(commit_url='https://huggingface.co/datasets/ty-kim/sql_translator/commit/58307faa0176121529604c689adcfbb288d3afe9', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='58307faa0176121529604c689adcfbb288d3afe9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ty-kim/sql_translator', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ty-kim/sql_translator'), pr_revision=None, pr_num=None)

In [30]:
print(card)

---
language:
- en
license: mit
size_categories:
- 10K<n<100K
pretty_name: Text-to-SQL Conversations
tags:
- text-to-sql
- synthetic
---

# Dataset Card for Text-to-SQL Conversations

<!-- Provide a quick summary of the dataset. -->



## Dataset Details

### Dataset Description

<!-- Provide a longer summary of what this dataset is. -->



- **Curated by:** [More Information Needed]
- **Funded by [optional]:** [More Information Needed]
- **Shared by [optional]:** [More Information Needed]
- **Language(s) (NLP):** ['en']
- **License:** mit

### Dataset Sources [optional]

<!-- Provide the basic links for the dataset. -->

- **Repository:** [More Information Needed]
- **Paper [optional]:** [More Information Needed]
- **Demo [optional]:** [More Information Needed]

## Uses

<!-- Address questions around how the dataset is intended to be used. -->

### Direct Use

<!-- This section describes suitable use cases for the dataset. -->

[More Information Needed]

### Out-of-Scope Use

<!-- Thi

In [34]:
len(dataset_dict["train"])

10500

In [ ]:
# 추후로 로드할 때 commit id 값을 이용해서 특정 버전을 로드할 수 있음
from datasets import load_dataset
ds = load_dataset(repo, split="train", revision="9b865ef")  # 특정 커밋 스냅숏

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForImageTextToText, BitsAndBytesConfig

In [21]:
model_id = "google/gemma-3-1b-pt"

if model_id == "google/gemma-3-1b-pt":
    model_class = AutoModelForCausalLM
else:
    model_class = AutoModelForImageTextToText

In [22]:
# Check if GPU benefits from bfloat16
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

In [23]:
torch_dtype

torch.bfloat16

In [24]:
# Define model init arguments
model_kwargs = dict(
    attn_implementation="eager", # Use "flash_attention_2" when running on Ampere or newer GPU
    torch_dtype=torch_dtype, # What torch dtype to use, defaults to auto
    device_map="auto", # Let torch decide how to load the model
)
